In [1]:
import contextlib
import importlib.util
import os
import sys
from itertools import cycle
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

TRAIN_DIR = Path.cwd() if Path.cwd().name == "train" else Path("/home/sunrise/orbitwars/pantheow/experimental_arch/train")
if str(TRAIN_DIR) not in sys.path:
    sys.path.insert(0, str(TRAIN_DIR))

from orbit_wars_engine import OrbitWarsEngine
from orbit_wars_model import encode_obs


In [7]:
from constants import (
    ACTION_CHOICES_PER_SOURCE,
    ACTIONS_DIM,
    GLOBAL_DIM,
    LAUNCH_GATE_CHOICES,
    NUM_FRAMES,
    PLANET_SLOTS,
    TOKEN_DIM,
)
from features import policy_action_mask


In [8]:
env = OrbitWarsEngine(num_players=2)

In [9]:
seed = np.random.randint(0, 100_000_000)
obs = env.reset(seed)['observations']

In [10]:
encoded_obs = encode_obs(obs[0], 0)  # for player 0
encoded_obs.keys()

dict_keys(['planet_ids', 'num_planets', 'frame_offsets', 'frame_planets', 'tokens', 'tokens_shape', 'globals', 'globals_shape', 'presence', 'presence_shape', 'turns', 'turns_shape', 'angles', 'angles_shape', 'mask', 'mask_shape', 'ship_counts', 'ship_counts_shape', 'reachable_mask', 'reachable_mask_shape'])

In [ ]:
global_features = torch.from_numpy(encoded_obs['globals'])
planet_tokens = torch.from_numpy(encoded_obs['tokens'].reshape(NUM_FRAMES, PLANET_SLOTS, TOKEN_DIM))
planet_presence = torch.from_numpy(encoded_obs['presence'].reshape(NUM_FRAMES, PLANET_SLOTS))
distances = torch.from_numpy(encoded_obs['turns'].reshape(NUM_FRAMES, PLANET_SLOTS, PLANET_SLOTS, ACTIONS_DIM))
raw_actions_mask = torch.from_numpy(encoded_obs['mask'].reshape(PLANET_SLOTS, PLANET_SLOTS, ACTIONS_DIM))
valid_actions_mask = torch.from_numpy(policy_action_mask(encoded_obs))
launch_angles = torch.from_numpy(encoded_obs['angles'].reshape(PLANET_SLOTS, PLANET_SLOTS, ACTIONS_DIM))
action_ship_counts = torch.from_numpy(encoded_obs['ship_counts'].reshape(PLANET_SLOTS, PLANET_SLOTS, ACTIONS_DIM))
reachable_mask = torch.from_numpy(encoded_obs['reachable_mask'].reshape(NUM_FRAMES, PLANET_SLOTS, PLANET_SLOTS, ACTIONS_DIM))


In [ ]:
from arch import (
    D_MODEL,
    EntityTransformer,
    GalaxyNet,
    GlobalEncoder,
    PlanetEncoder,
    PolicyHead,
    ValueHead,
)


In [ ]:
model_obs = {
    "globals": global_features.unsqueeze(0),
    "tokens": planet_tokens.unsqueeze(0),
    "presence": planet_presence.unsqueeze(0),
    "turns": distances.unsqueeze(0),
    "reachable_mask": reachable_mask.unsqueeze(0),
    "valid_actions_mask": valid_actions_mask.unsqueeze(0),
}

galaxy_net = GalaxyNet()
value, flat_logits = galaxy_net(model_obs)
value.shape, flat_logits.shape


In [ ]:
planet_encoder = PlanetEncoder()
global_encoder = GlobalEncoder()
transformer = EntityTransformer()
value_head = ValueHead()
policy_head = PolicyHead()

planet_tokens_encoded = planet_encoder(model_obs["tokens"].float()[:, 0])
global_token = global_encoder(model_obs["globals"].float())
planet_encoded, global_encoded = transformer(
    planet_tokens_encoded,
    global_token,
    model_obs["presence"].float()[:, 0],
)

value = value_head(global_encoded, model_obs["globals"].float())
logits = policy_head(planet_encoded, global_encoded, model_obs["globals"].float(), model_obs)
value.shape, logits.shape


In [357]:
# Per-source policy layout: one launch gate categorical, then one target categorical.
# The target categorical is ignored by decode_action when gate==noop.
gate_logits = flat_logits[:, :LAUNCH_GATE_CHOICES]
target0_start = LAUNCH_GATE_CHOICES
target0_end = target0_start + PLANET_SLOTS
target_logits_source0 = flat_logits[:, target0_start:target0_end]

gate_probs = F.softmax(gate_logits, dim=-1)
target_probs_source0 = F.softmax(target_logits_source0, dim=-1)
gate_probs.shape, target_probs_source0.shape, valid_actions_mask.shape


tensor([[nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan],
        ...,
        [nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan]], grad_fn=<SoftmaxBackward0>)

In [338]:
h = torch.randn(PLANET_SLOTS, D_MODEL)
h.unsqueeze(1).expand(PLANET_SLOTS, PLANET_SLOTS, -1).shape


tensor([[[ 1.0265, -0.5188, -0.1022,  ...,  0.4857, -0.2457, -0.3435],
         [-0.0531,  1.0216, -0.2858,  ..., -0.1946,  0.3140,  0.2594],
         [ 0.4319, -0.5107,  0.7150,  ..., -0.3964,  0.4322, -0.0792],
         ...,
         [ 1.7345,  0.8501, -0.4626,  ..., -0.2404,  0.8107,  1.1856],
         [-1.7627, -1.0018,  1.2368,  ...,  0.7324, -1.3661,  0.5996],
         [-0.9521, -0.4527, -0.6780,  ..., -1.6143,  0.4245, -1.4192]],

        [[ 1.0265, -0.5188, -0.1022,  ...,  0.4857, -0.2457, -0.3435],
         [-0.0531,  1.0216, -0.2858,  ..., -0.1946,  0.3140,  0.2594],
         [ 0.4319, -0.5107,  0.7150,  ..., -0.3964,  0.4322, -0.0792],
         ...,
         [ 1.7345,  0.8501, -0.4626,  ..., -0.2404,  0.8107,  1.1856],
         [-1.7627, -1.0018,  1.2368,  ...,  0.7324, -1.3661,  0.5996],
         [-0.9521, -0.4527, -0.6780,  ..., -1.6143,  0.4245, -1.4192]],

        [[ 1.0265, -0.5188, -0.1022,  ...,  0.4857, -0.2457, -0.3435],
         [-0.0531,  1.0216, -0.2858,  ..., -0

In [ ]:
edge_mask = reachable_mask[:, :, :, :].transpose(1, 2).bool().any(dim=-1)
edge_mask.shape, edge_mask.float().mean()
